# kinematics: SO-101 forward and inverse kinematics

Library notebook for `gambit_arm`. Forward kinematics (FK) of the SO-ARM101 built directly
from the joint origins and axes in `gambit_description/urdf/so101.urdf.xacro`, a numeric
Jacobian, and a damped-least-squares (DLS) inverse kinematics (IK) for "put the gripper tip
here, approaching from this direction".

Milestone: *Hardware and execution* (pick-and-place needs joint targets for each square).
The C++ mirror is `gambit_cpp/libs/kinematics` (FK implemented, IK TODO); its test checks the
same zero-pose numbers as the self-check below.

**TODO**
- [x] parse the URDF joint chain base_link -> gripper_frame_link (with a hardcoded fallback)
- [x] FK (4x4 tool pose), numeric Jacobian
- [x] simple DLS IK for position + approach direction
- [ ] joint-limit handling better than clipping (e.g. weighted DLS or null-space push away
      from limits); seeding strategy (elbow-up branch) so IK never flips configuration
- [ ] top-down orientation constraint: approach within `max_tilt_deg` of vertical, choose the
      tilt per square (far ranks need tilt), and pick wrist_roll so the fingers align with
      the board's files/ranks
- [ ] compare FK against the real arm after calibration (tape measure / repeatability run)
- [ ] precompute and cache IK for all 64 squares at hover and grasp height

In [ ]:
import math
import xml.etree.ElementTree as ET
import numpy as np

%run ./square_map.ipynb

## Joint chain from the URDF

URDF convention: each joint's transform from parent link to child link is

$$T = \mathrm{Trans}(xyz)\;R_z(y)\,R_y(p)\,R_x(r)\;\mathrm{Rot}(\text{axis}, q)$$

We follow the chain `base_link -> shoulder_link -> upper_arm_link -> lower_arm_link ->
wrist_link -> gripper_link -> gripper_frame_link`. The last joint (`gripper_frame_joint`) is
fixed and defines the **tool frame**: its z axis is the approach direction (pointing out of
the fingers), its origin is the point between the fingertips we call "the gripper tip".

`HARDCODED_CHAIN` holds the same numbers copied from the xacro (Nov 2026), used if parsing
fails (e.g. the description package is missing in Colab).

In [ ]:
ARM_JOINTS = ["shoulder_pan", "shoulder_lift", "elbow_flex", "wrist_flex", "wrist_roll"]
TOOL_LINK = "gripper_frame_link"

# (name, type, xyz, rpy, axis, lower, upper) from gambit_description/urdf/so101.urdf.xacro
HARDCODED_CHAIN = [
    ("shoulder_pan",  "revolute", (0.0388353, -8.97657e-09, 0.0624), (3.14159, 4.18253e-17, -3.14159), (0, 0, 1), -1.91986, 1.91986),
    ("shoulder_lift", "revolute", (-0.0303992, -0.0182778, -0.0542), (-1.5708, -1.5708, 0), (0, 0, 1), -1.74533, 1.74533),
    ("elbow_flex",    "revolute", (-0.11257, -0.028, 1.73763e-16), (-3.63608e-16, 8.74301e-16, 1.5708), (0, 0, 1), -1.69, 1.69),
    ("wrist_flex",    "revolute", (-0.1349, 0.0052, 3.62355e-17), (4.02456e-15, 8.67362e-16, -1.5708), (0, 0, 1), -1.65806, 1.65806),
    ("wrist_roll",    "revolute", (5.55112e-17, -0.0611, 0.0181), (1.5708, 0.0486795, 3.14159), (0, 0, 1), -2.74385, 2.84121),
    ("gripper_frame_joint", "fixed", (-0.0079, -0.000218121, -0.0981274), (0, 3.14159, 0), (0, 0, 0), 0.0, 0.0),
]

URDF_PATH = REPO_ROOT / "gambit_description" / "urdf" / "so101.urdf.xacro"

def parse_chain(path=URDF_PATH, base="base_link", tip=TOOL_LINK):
    """Joint chain base -> tip as a list of (name, type, xyz, rpy, axis, lower, upper).
    Only <joint> elements with a type attribute are kinematic (the ros2_control block reuses
    the <joint> tag without one)."""
    root = ET.parse(path).getroot()
    by_child = {}
    for j in root.iter("joint"):
        if j.get("type") not in ("revolute", "continuous", "prismatic", "fixed"):
            continue
        o, a, lim = j.find("origin"), j.find("axis"), j.find("limit")
        fl = lambda s: tuple(float(v) for v in s.split())
        try:
            spec = (j.get("name"), j.get("type"),
                    fl(o.get("xyz", "0 0 0")), fl(o.get("rpy", "0 0 0")),
                    fl(a.get("xyz")) if a is not None else (0, 0, 0),
                    float(lim.get("lower")) if lim is not None else 0.0,
                    float(lim.get("upper")) if lim is not None else 0.0,
                    j.find("parent").get("link"))
        except ValueError:          # xacro expressions such as $(arg base_xyz) (world_to_base)
            continue
        by_child[j.find("child").get("link")] = spec
    chain, link = [], tip
    while link != base:
        if link not in by_child:
            raise ValueError(f"no joint leads to {link}")
        *spec, parent = by_child[link]
        chain.append(tuple(spec))
        link = parent
    return chain[::-1]

try:
    CHAIN = parse_chain()
    CHAIN_SOURCE = str(URDF_PATH.relative_to(REPO_ROOT))
except Exception as e:                      # missing file or xacro features we do not parse
    print("URDF parse failed, using HARDCODED_CHAIN:", e)
    CHAIN, CHAIN_SOURCE = HARDCODED_CHAIN, "HARDCODED_CHAIN"

JOINT_LOWER = np.array([c[5] for c in CHAIN if c[1] != "fixed"])
JOINT_UPPER = np.array([c[6] for c in CHAIN if c[1] != "fixed"])
[c[0] for c in CHAIN], CHAIN_SOURCE

## Forward kinematics

Multiply the joint transforms. `fk(q)` returns the 4x4 tool pose in the arm base frame
(= world frame, since `arm.base_xyz` is the origin; add it if the arm moves).

Sign conventions worth knowing (they surprised us): the shoulder_pan origin flips its z axis
(rpy pi, 0, -pi), so **positive pan swings the tool toward -y** (the arm's right). At q = 0
the arm points straight out along +x, horizontal.

In [ ]:
def rpy_matrix(r, p, y):
    cr, sr, cp, sp, cy, sy = math.cos(r), math.sin(r), math.cos(p), math.sin(p), math.cos(y), math.sin(y)
    Rx = np.array([[1, 0, 0], [0, cr, -sr], [0, sr, cr]])
    Ry = np.array([[cp, 0, sp], [0, 1, 0], [-sp, 0, cp]])
    Rz = np.array([[cy, -sy, 0], [sy, cy, 0], [0, 0, 1]])
    return Rz @ Ry @ Rx

def axis_angle_matrix(axis, q):
    """Rodrigues rotation about a unit axis."""
    k = np.asarray(axis, float)
    k = k / np.linalg.norm(k)
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + math.sin(q) * K + (1 - math.cos(q)) * K @ K

def homogeneous(R=np.eye(3), t=(0, 0, 0)):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T

# Precompute the fixed origin transforms once.
_ORIGINS = [homogeneous(rpy_matrix(*c[3]), c[2]) for c in CHAIN]

def fk_all(q, base_xyz=None):
    """List of 4x4 frames after each joint in CHAIN (last = tool). q: 5 arm joint angles (rad)."""
    q = np.asarray(q, float)
    T = homogeneous(t=base_xyz if base_xyz is not None else BOARD["arm"]["base_xyz"])
    frames, i = [], 0
    for c, O in zip(CHAIN, _ORIGINS):
        T = T @ O
        if c[1] != "fixed":
            T = T @ homogeneous(axis_angle_matrix(c[4], q[i]))
            i += 1
        frames.append(T)
    return frames

def fk(q):
    """4x4 pose of the tool frame (gripper tip; z axis = approach direction)."""
    return fk_all(q)[-1]

def tool_position(q):
    return fk(q)[:3, 3]

def approach_vector(q):
    return fk(q)[:3, 2]

def tilt_from_vertical_deg(q):
    """Angle between the approach vector and straight down (0 = perfectly top-down)."""
    a = approach_vector(q)
    return math.degrees(math.acos(np.clip(-a[2], -1, 1)))

PARK = np.array(BOARD["arm"]["park"])
np.round(tool_position(np.zeros(5)), 4), np.round(tool_position(PARK), 4)

## Numeric Jacobian

Central differences of the 6-vector task `[position; approach]` with respect to q. Cheap
enough for 5 joints and avoids deriving the analytic Jacobian (do that later in C++ if speed
matters).

In [ ]:
def task_vector(q):
    T = fk(q)
    return np.r_[T[:3, 3], T[:3, 2]]

def numeric_jacobian(q, eps=1e-6):
    q = np.asarray(q, float)
    J = np.zeros((6, len(q)))
    for j in range(len(q)):
        d = np.zeros(len(q))
        d[j] = eps
        J[:, j] = (task_vector(q + d) - task_vector(q - d)) / (2 * eps)
    return J

## Inverse kinematics (damped least squares)

Goal: tip at `p*` with approach direction `a*`. Error `e = [p* - p(q); w (a* - a(q))]`
(the weight `w` trades millimeters for radians). DLS step:

$$\Delta q = J^T (J J^T + \lambda^2 I)^{-1} e$$

`lambda` keeps steps bounded near singularities (arm fully stretched or folded). After each
step we clip to the joint limits; that is the crude part (TODO above).

With 5 joints and 5 meaningful constraints (3 position + 2 for a direction), wrist_roll is
nearly free: it only spins the fingers about the approach axis. A good default target is
`approach_from_pitch(p, pitch)`: tilt toward the target from straight down, in the vertical
plane through the arm base.

In [ ]:
def approach_from_pitch(target, pitch_rad):
    """Unit approach vector pointing from above toward `target`: pitch = angle below horizontal
    (pi/2 = straight down), leaning away from the arm base in the vertical plane through it."""
    xy = np.asarray(target[:2], float)
    u = xy / (np.linalg.norm(xy) + 1e-12)
    return np.r_[math.cos(pitch_rad) * u, -math.sin(pitch_rad)]

def ik_dls(target, approach=None, q0=None, pitch=None, iters=300, lam=0.01, w_orient=0.05,
           pos_tol=1e-4, orient_tol=2e-3, limits=True, lower=None, upper=None):
    """Damped-least-squares IK.
    target: (3,) tip position in m; approach: (3,) unit vector or None (position only).
    pitch: instead of a fixed approach, ask for this angle below horizontal (pi/2 = straight
    down) and let the horizontal direction follow the arm's own vertical plane (re-evaluated
    every iteration). pitch > pi/2 leans the approach back toward the arm. With only 4 joints that move the approach in that plane, a fixed
    approach that is not in the plane is unreachable, so prefer `pitch` for top-down grasps.
    q0: seed (defaults to a bent pose above the board). lower/upper: joint limits to clip to
    (default: the URDF limits; safety.ipynb passes its tighter ones). Returns (q, info) with
    info = {converged, pos_err, orient_err, iters}."""
    q = np.array([0.0, 0.2, 0.3, 1.4, 0.0] if q0 is None else q0, float)
    p_t = np.asarray(target, float)
    a_t = None if approach is None else np.asarray(approach, float) / np.linalg.norm(approach)
    if pitch is not None:
        a_t = approach_from_pitch(p_t, pitch)
    lower = JOINT_LOWER if lower is None else lower
    upper = JOINT_UPPER if upper is None else upper
    rows = 3 if a_t is None else 6
    W = np.diag([1, 1, 1, w_orient, w_orient, w_orient][:rows])
    for it in range(iters):
        v = task_vector(q)
        if pitch is not None:
            # The arm's vertical plane is the x-z plane rotated by the pan angle (at q = 0 the
            # arm lies along +x; +pan swings toward -y), so its forward direction is:
            h = np.array([math.cos(q[0]), -math.sin(q[0])])
            a_t = np.r_[math.cos(pitch) * h, -math.sin(pitch)]
        e = (p_t - v[:3]) if a_t is None else np.r_[p_t - v[:3], a_t - v[3:]]
        pe = np.linalg.norm(e[:3])
        oe = 0.0 if a_t is None else np.linalg.norm(e[3:])
        if pe < pos_tol and oe < orient_tol:
            return q, dict(converged=True, pos_err=pe, orient_err=oe, iters=it)
        J = W @ numeric_jacobian(q)[:rows]
        dq = J.T @ np.linalg.solve(J @ J.T + lam ** 2 * np.eye(rows), W @ e)
        q = q + dq
        if limits:
            # TODO: replace clipping with proper limit handling (see the TODO list).
            q = np.clip(q, lower, upper)
    return q, dict(converged=False, pos_err=pe, orient_err=oe, iters=iters)

def ik_top_down(target, tilt_deg=0.0, q0=None, **kw):
    """IK for a (nearly) top-down approach at `target`.
    Positive tilt leans the approach away from the arm base, negative toward it.
    TODO (partly done in pick_place.plan_joint_trajectory, which tries a list of tilts): choose tilt automatically per square within BOARD['arm']['max_tilt_deg'] (try 0, then
    increase until IK converges inside limits), constrain wrist_roll so the fingers close along
    a file or rank, and return None (not a bad q) when no solution exists."""
    if abs(tilt_deg) > BOARD["arm"]["max_tilt_deg"]:
        raise ValueError("tilt exceeds arm.max_tilt_deg in board.yaml")
    return ik_dls(target, pitch=math.radians(90 - tilt_deg), q0=q0, **kw)

### Reachability over the board (quick look)

For each square: does top-down IK converge at hover height? (Prints a grid; `.` = converged,
`x` = not.) This is the starting point for the per-square tilt TODO.

In [ ]:
def reachability_grid(height=None, tilt_deg=0.0):
    h = BOARD["arm"]["hover_height"] if height is None else height
    rows = []
    for r in range(7, -1, -1):
        line = ""
        for f in range(8):
            q, info = ik_top_down(hover_point(square_name(f, r)) + [0, 0, h - BOARD["arm"]["hover_height"]], tilt_deg)
            line += "." if info["converged"] else "x"
        rows.append(f"{r + 1} {line}")
    rows.append("  abcdefgh")
    return "\n".join(rows)

# print(reachability_grid())          # ~2 s; uncomment to look

## Self-check

In [ ]:
assert [c[0] for c in CHAIN] == ARM_JOINTS + ["gripper_frame_joint"]
assert CHAIN_SOURCE != "HARDCODED_CHAIN" or not URDF_PATH.exists()   # the parser really ran
for got, ref in zip(CHAIN, HARDCODED_CHAIN):        # parsed URDF still matches the copied values
    assert np.allclose(got[2], ref[2]) and np.allclose(got[3], ref[3]) and np.isclose(got[5], ref[5])
T0 = fk(np.zeros(5))
assert np.all(np.isfinite(T0))
assert np.allclose(T0[:3, :3] @ T0[:3, :3].T, np.eye(3), atol=1e-5)
assert np.allclose(T0[:3, 3], [0.3914, 0.0, 0.2265], atol=5e-4)   # same numbers as gambit_cpp test
assert tool_position([0.5, 0, 0, 0, 0])[1] < 0                     # +pan swings toward -y
J = numeric_jacobian([0.1, -0.3, 0.4, 1.2, 0.0])
assert J.shape == (6, 5) and np.all(np.isfinite(J))
# IK round trip: FK of a known pose -> IK from a perturbed seed -> same tip and approach
q_true = np.array([0.1, -0.3, 0.4, 1.2, 0.0])
q_sol, info = ik_dls(tool_position(q_true), approach_vector(q_true), q0=q_true + 0.1)
assert info["converged"], info
assert np.linalg.norm(tool_position(q_sol) - tool_position(q_true)) < 1e-3
# top-down at e4 hover height is reachable
q_e4, info = ik_top_down(hover_point("e4"))
assert info["converged"] and tilt_from_vertical_deg(q_e4) < 1.0, info
print("kinematics OK")